## IMPORT LIBRAIRIES

In [27]:
import os
from pathlib import Path
import time
import joblib
import glob
from colorama import Fore, Style

from scipy.stats import randint, uniform, loguniform
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from keras import Model, models

### Supers functions

In [24]:
def save_model(model, path="ml_logic"):
    """
    Save the model with timestamp
    - works for keras or other models
    """

    timestamp = time.strftime("%Y%m%d-%H%M%S")

    if isinstance(model, Model):
        model_path = f"{path}/models/{timestamp}.keras"
        model.save(model_path)

    else:
        model_path = f"{path}/models/{timestamp}.pkl"
        joblib.dump(model, model_path)

    print(f"✅ Model saved at {model_path}")

def load_model(path="ml_logic"):
    """
    Load the model
    - works for different models
    """
    print(Fore.BLUE + "\nLoad latest model from local registry..." + Style.RESET_ALL)

    local_model_directory = os.path.join(path, "models")
    local_model_paths = glob.glob(f"{local_model_directory}/*")

    if not local_model_paths:
        return None

    most_recent_model_path_on_disk = sorted(local_model_paths)[-1]

    print(Fore.BLUE + "\nLoad latest model from disk..." + Style.RESET_ALL)

    if most_recent_model_path_on_disk.endswith(".keras") or most_recent_model_path_on_disk.endswith(".h5"):
        model = models.load_model(most_recent_model_path_on_disk)

    elif most_recent_model_path_on_disk.endswith(".json"):
        model = XGBClassifier()
        model.load_model(most_recent_model_path_on_disk)

    else:  # .pkl
        model = joblib.load(most_recent_model_path_on_disk)

    print("✅ Model loaded from local disk")

    return model


def add_noise_to_dataset(df, columns=None, noise_fraction=0.12, possible_values_dict=None):
    """
    Ajoute du bruit à plusieurs colonnes d'un DataFrame d'un seul coup.

    :param df: Le DataFrame de test original.
    :param columns: Liste des colonnes à bruiter. Si None, applique à toutes les colonnes.
    :param noise_fraction: La proportion de données à modifier par colonne (ex: 0.05).
    :param possible_values_dict: (Optionnel) Un dictionnaire {nom_colonne: [valeurs_possibles]}.
    :return: Un nouveau DataFrame avec le bruit ajouté.
    """
    df_noisy = df.copy()

    # Si aucune colonne n'est spécifiée, on prend toutes les colonnes du dataset
    if columns is None:
        columns = df_noisy.columns

    n_rows = len(df_noisy)
    n_noise = int(n_rows * noise_fraction)

    for col in columns:
        # 1. Déterminer les valeurs possibles pour cette colonne spécifique
        if possible_values_dict and col in possible_values_dict:
            possible_values = possible_values_dict[col]
        else:
            # Détection automatique : on prend les valeurs uniques existantes dans la colonne
            possible_values = df_noisy[col].dropna().unique()

        # Si la colonne est vide ou n'a qu'une seule valeur possible, on l'ignore
        if len(possible_values) <= 1:
            continue

        # 2. Sélectionner les lignes à modifier (indices au hasard)
        noise_indices = np.random.choice(df_noisy.index, size=n_noise, replace=False)

        # 3. Générer les nouvelles valeurs aléatoires parmi les choix possibles pour cette colonne
        random_new_values = np.random.choice(possible_values, size=n_noise)

        # 4. Appliquer le bruit
        df_noisy.loc[noise_indices, col] = random_new_values

    return df_noisy

## IMPORT DATASETS

In [3]:
df_primary = pd.read_csv("../data/table_dataset/primary_data.csv", sep=";")
df_secondary = pd.read_csv("../data/table_dataset/secondary_data.csv", sep=";")
df_mushnames = pd.read_csv("../data/table_dataset/species_names.csv", sep =";")

### Cleaning data tabular 

#### Data for edible classification

In [4]:
# Species name from primary, add to secondary
primary_name_df = df_primary[['family','name']]
df_primname_rep = primary_name_df.loc[primary_name_df.index.repeat(353)].reset_index(drop=True)

# Clean names
data_secondary_labelled = pd.concat([df_secondary, df_primname_rep], axis=1)
data_secondary_labelled['family'] = data_secondary_labelled['family'].str.replace(" Family", "", regex=False)
#
data_secondary_labelled['Common Name'] = data_secondary_labelled["family"] + " " + data_secondary_labelled["name"]
data_secondary_labelled

# Merge to scientific names
data_merge_scname = data_secondary_labelled.merge(df_mushnames, how='left', on='Common Name')
data_tabular_final = data_merge_scname.drop(columns=['family','name','Common Name'])
data_tabular_final.columns = data_tabular_final.columns.str.replace('-', '_').str.replace(' ', '_').str.lower()
data_tabular_final["class"] = (data_tabular_final["class"] == "p").astype(int)

# Feature selection
data_tabular_final = data_tabular_final.drop(columns=['cap_surface','ring_type',
                                                      'cap_diameter','stem_height','stem_width'])

# Randomly ordered df
data_tabular_rdm = data_tabular_final.sample(frac=1, random_state=3).reset_index(drop=True)

# Apply a noise to the data : TOO perfect is not good
data_tab_rdm_noise = add_noise_to_dataset(data_tabular_rdm, noise_fraction=0.12)

data_tab_rdm_noise.info()

<class 'pandas.DataFrame'>
RangeIndex: 61069 entries, 0 to 61068
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   class                 61069 non-null  int64
 1   cap_shape             61069 non-null  str  
 2   cap_color             61069 non-null  str  
 3   does_bruise_or_bleed  61069 non-null  str  
 4   gill_attachment       52367 non-null  str  
 5   gill_spacing          39078 non-null  str  
 6   gill_color            61069 non-null  str  
 7   stem_root             15710 non-null  str  
 8   stem_surface          27557 non-null  str  
 9   stem_color            61069 non-null  str  
 10  veil_type             3177 non-null   str  
 11  veil_color            13841 non-null  str  
 12  has_ring              61069 non-null  str  
 13  spore_print_color     12879 non-null  str  
 14  habitat               61069 non-null  str  
 15  season                61069 non-null  str  
 16  scientific_name

In [1]:
#data_tabular_final.sample(frac=1, random_state=3).reset_index(drop=True)

#### Data for species classification

In [5]:
## Liste champignons (via noms scientifiques) dans les images
path_edible = "../data/image_dataset/edible"
path_poisonous = "../data/image_dataset/poisonous"

# Liste des sous-dossiers
list_edible = [f for f in os.listdir(path_edible)
                 if os.path.isdir(os.path.join(path_edible, f))]
list_poisonous = [f for f in os.listdir(path_poisonous)
                 if os.path.isdir(os.path.join(path_poisonous, f))]

# Création du DataFrame
df1 = pd.DataFrame(list_edible, columns=["scientific_name"])
df1['type'] = "edible"
df2 = pd.DataFrame(list_poisonous, columns=["scientific_name"])
df2['type'] = "poisonous"
df_concat = pd.concat([df1, df2], ignore_index=True)
df_concat['scientific_name'] = df_concat['scientific_name'].str.replace("_", " ", regex=False).str.replace("-", " ", regex=False)

# garder que le tabulaire dont l'espèce est présente dans les données d'image
data_tabular_image = data_tab_rdm_noise.merge(df_concat, how='inner', on='scientific_name')
data_tabular_image

,class,cap_shape,cap_color,does_bruise_or_bleed,gill_attachment,gill_spacing,gill_color,stem_root,stem_surface,stem_color,veil_type,veil_color,has_ring,spore_print_color,habitat,season,scientific_name,type
0,1,x,y,f,x,NaN,w,c,NaN,w,NaN,NaN,f,NaN,d,a,Coprinus comatus,edible
1,1,b,g,f,NaN,c,n,NaN,NaN,n,NaN,NaN,t,NaN,g,a,Psilocybe semilanceata,poisonous
2,0,o,n,f,a,d,y,NaN,k,o,NaN,NaN,f,NaN,d,u,Flammulina velutipes,edible
3,1,f,n,f,d,f,y,b,NaN,k,NaN,u,f,NaN,l,w,Ampulloclitocybe clavipes,poisonous
4,0,f,g,f,a,c,w,s,k,w,NaN,NaN,f,g,d,w,Clitocybe nebularis,poisonous
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16905,0,p,w,f,e,f,p,NaN,s,w,NaN,n,t,NaN,d,u,Coprinus comatus,edible
16906,0,s,p,t,e,c,o,NaN,NaN,w,NaN,e,f,NaN,u,a,Lactarius deliciosus,edible
16907,1,x,y,f,a,c,y,NaN,g,o,NaN,u,f,u,h,a,Suillus luteus,edible
16908,1,x,r,f,NaN,f,w,NaN,NaN,w,u,w,t,NaN,d,a,Amanita phalloides,poisonous


### Train Test Split both datas

In [6]:
# EDIBLE CLASSIFICATION
# prepare X and y

X = data_tab_rdm_noise.drop(columns=['class','gill_spacing','stem_root','stem_surface',
                                     'veil_type','veil_color','spore_print_color','scientific_name'])
y = data_tab_rdm_noise['class']

# TTS

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=3)


In [7]:
# SPECIES CLASSIFICATION
# prepare X and y

X_tabimage = data_tabular_image.drop(columns=['class','gill_spacing','stem_root','stem_surface',
                                     'veil_type','veil_color','spore_print_color','scientific_name'])
y_tabimage = data_tabular_image['scientific_name']

# TTS

X_tabimage_train, X_tabimage_test, y_tabimage_train, y_tabimage_test = train_test_split(X_tabimage,
                                                                            y_tabimage,
                                                                            test_size=0.3,
                                                                            random_state=3)

### Preprocess pipeline

In [8]:
# pipeline num and cat
num_transformer = make_pipeline(SimpleImputer(strategy="median"), MinMaxScaler())
cat_transformer = make_pipeline(SimpleImputer(strategy='constant', fill_value='u'),
                                OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False))

# preprocess all
preproc_basic = make_column_transformer(
    (num_transformer, make_column_selector(dtype_include=np.number)),
    (cat_transformer, make_column_selector(dtype_exclude=np.number)),
    remainder='drop'
).set_output(transform="pandas")


## XGBOOST BASELINE - Edibility classification

In [9]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42)

pipe_baseline = make_pipeline(preproc_basic, model)
pipe_baseline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('xgbclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline-1', ...), ('pipeline-2', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of th

In [10]:
score_baseline = cross_val_score(pipe_baseline, X_train, y_train, cv=5, scoring='recall').mean()
score_baseline

np.float64(0.8629661662149406)

In [11]:
param_dist = {
    "xgbclassifier__max_depth": randint(3, 8),
    "xgbclassifier__learning_rate": loguniform(0.01, 0.2),
    "xgbclassifier__n_estimators": randint(100, 501),
    "xgbclassifier__subsample": uniform(0.6, 0.4),
    "xgbclassifier__colsample_bytree": uniform(0.6, 0.4),
    "xgbclassifier__gamma": uniform(0, 0.5)
}

search = RandomizedSearchCV(
    pipe_baseline,
    param_distributions=param_dist,
    n_iter=150,
    cv=5,
    scoring="recall",  # on optimise le recall
    n_jobs=-1,
    verbose=1,
    random_state=42
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best recall:", search.best_score_)

Fitting 5 folds for each of 150 candidates, totalling 750 fits
Best params: {'xgbclassifier__colsample_bytree': np.float64(0.836965827544817), 'xgbclassifier__gamma': np.float64(0.023225206359998862), 'xgbclassifier__learning_rate': np.float64(0.061721159481070736), 'xgbclassifier__max_depth': 7, 'xgbclassifier__n_estimators': 428, 'xgbclassifier__subsample': np.float64(0.6260206371941118)}
Best recall: 0.8663769619181856


In [ ]:
best_model = search.best_estimator_
save_model(best_model, '../ml_logic')


✅ Model saved at ../ml_logic/models/20260312-164502.pkl


In [29]:
loaded_model_hehe = load_model('../ml_logic')

loaded_model_hehe.predict(X_test)


Load latest model from local registry...

Load latest model from disk...
✅ Model loaded from local disk


array([1, 1, 1, ..., 1, 1, 1], shape=(18321,))

#### Importance of features, can be done with ShapValues

In [ ]:
#pipe_baseline.fit(X_train, y_train)
#
#importances = pipe_baseline[-1].feature_importances_
#features = pipe_baseline[0].get_feature_names_out()
#
#feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)
#

In [1]:
#dfimp = pd.DataFrame({
#    "importances": importances,
#    "features": features
#})
#dfimp["features"] = dfimp["features"].str.replace(r".*__", "", regex=True).str[:-2]
#
#dfimp.groupby("features", as_index=False)["importances"]\
#      .sum()\
#      .sort_values(by="importances", ascending=False)

In [38]:
#pipe_baseline.fit(X_train, y_train)
#pipe_baseline.predict(X_test)

## RandomForest BASELINE - Edibility classification

In [30]:
#model = RandomForestClassifier()
#pipe_rf = make_pipeline(preproc_basic, model)
#score_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring='accuracy').mean()
#score_rf